# Survivor Data Set: An Exploration
## Project Overview

We seek to understand the underlying themes in the reality TV gameshow "Survivor". The show is currently airing it's 50th season, but the gameplay we see in the later seasons is very different from the early seasons of the show. Survivor is a gameshow about physical challenges and mental strength, but fundamentally it is a show about relationships with others. In a game where the goal is to "Outwit, Outplay, Outlast", what does this actually translate to? In order to win you need the votes of the jury, which is made up of the players you had a hand in voting off of the tribe.

The thing that makes Survivor so interesting is how different every season is, and how the game is always evolving. In early seasons, loyalty and morals were valued, and the gameplay was relatively simple. As time went on, players got more conniving, and blindsides and backstabbing became the norm. But what is most interesting is that as the gameplay changed, so did the mentality of the players. Playing fair isn't enough to win anymore. 

Through this data we hope to find answers to some of these questions:
- What does it take Outwit, Outplay, and Outlast?
- What decisions to winners make? 
- What decisions do losers make?
- How have the decisions made by winners and losers changed as the game has evolved?
- Are there trends that predict performance?
- Are there qualities that predict performance?
- Have there been changes in these trends/qualities over time?

## Data Description and Source
The original survivoR dataset is in R, [original data set](https://github.com/doehm/survivoR/tree/master/data). We will be using a modified version of this dataset that has been converted to CSV files found here: [Survivor Data Set](https://github.com/rfordatascience/tidytuesday/tree/master/data/2021/2021-06-01)

The original dataset has 23 R data files, while the dataset that's been converted to CSV only has 5 of these. 

The 5 csv files are:
- Summary
- Challanges
- Castaways
- Viewers
- Jury Votes


# Initial CSV Data
The sections below walk through each CSV in the same order: **Initial CSV exploration**, **Cleaning needs**, **Methods**, and **Results**.


## Setup: imports and loading

Import **pandas** and load all tables from the TidyTuesday CSV mirror.


In [ ]:
# Obligatory pandas import
import pandas as pd

In [ ]:
# Each table is available as a CSV from the TidyTuesday GitHub mirror
base_url = "https://raw.githubusercontent.com/rfordatascience/tidytuesday/master/data/2021/2021-06-01/"

# Load the main tables
summary_df     = pd.read_csv(base_url + "summary.csv")
challenges_df  = pd.read_csv(base_url + "challenges.csv")
castaways_df   = pd.read_csv(base_url + "castaways.csv")
viewers_df     = pd.read_csv(base_url + "viewers.csv")
jury_votes_df  = pd.read_csv(base_url + "jury_votes.csv")

## Table: Summary


### Initial CSV exploration

#### Context

#### Relevance:
- Primary data set for information on seasons, winners, and dates.
- Useful for determining winners and trends over time.

#### Size:
- 40 rows
- 19 columns
- 6.1 KB

#### Column descriptions (English):
- **Season Name**: The name of the season
- **Season**: The season number
- **Location**: The geographical location of the show
- **Country**: The country where the season takes place
- **Tribe Setup**: How players are divided into tribes (teams)
- **Full Name**: The name of the player
- **Winner**: The winner of the season
- **Runner Ups**: Second place
- **Final Vote**: The vote split for the winner
- **Time Slot**: Day and time of week when episodes aired
- **Premiered**: Date when the season premiered
- **Ended**: Date when the season ended
- **Filming Strated**: Date when filming started (note: this spelling matches the CSV column label)
- **Filming Ended**: Date when filming ended
- **Viewers Finale**: Number of viewers for the final episode
- **Viewers Reunion**: Number of viewers for the reunion (contestant retrospective)
- **Viewers Mean**: Average viewership for episodes
- **Rank**: Viewer final ranking of the contestant

**Note**: Actual column names and data types are shown below, along with a view of the first five rows.


#### Raw inspection


In [ ]:
summary_df.info()
summary_df.head()

### Cleaning needs

The following cells repeat `summary_df` inspection where useful, then apply parsing and type fixes already developed for this table.


_Cleaning steps (summary)_


### Methods

The following code cells implement the cleaning steps for `summary_df`.


In [ ]:
# Initial Dataframe
summary_df.info()
summary_df.head()

#### Initial status **above**
- the columns `premiered`, `ended`, `filming_started`, and `filming_ended` need to be converted to datetimes instead of strings
- the column `timeslot` should be converted to `weekday` and `air_time` columns
- the 2 null values in `viewers_mean` and `rank` columns need to be addressed


In [ ]:
# View the timeslot column entries
summary_df['timeslot'].value_counts()

In [ ]:
# Extract weekday and time text from timeslot
parts = summary_df["timeslot"].str.extract(r"(\w+)\s+(.+)")
summary_df["weekday"] = parts[0]
summary_df["air_time_raw"] = parts[1]

# Parse time text into a time value
summary_df["air_time"] = pd.to_datetime(
    summary_df["air_time_raw"].str.strip().str.lower(),
    format="%I:%M %p",
    errors="coerce"
).dt.time

# Drop the timeslot column and airtime_raw column
summary_df = summary_df.drop(columns=["timeslot", "air_time_raw"])

summary_df.info()

In [ ]:
# Convert the premiered, ended, filming_started, and filming_ended columns to datetime
summary_df['premiered'] = pd.to_datetime(summary_df['premiered'])
summary_df['ended'] = pd.to_datetime(summary_df['ended'])
summary_df['filming_started'] = pd.to_datetime(summary_df['filming_started'])
summary_df['filming_ended'] = pd.to_datetime(summary_df['filming_ended'])

# Show the updated dataframe
summary_df.info()

In [ ]:
# View the null values in viewers_mean and rank
summary_df[summary_df['viewers_mean'].isna() | summary_df['rank'].isna()]

`rank` and `viewers_mean` are missing in the same two rows. Since `rank` may be important for our modeling later, we removed rows with missing `rank` from the dataset. Otherwise, we could retain the full table and apply a filter when building a model.

If the NaN values were kept: 
- `rank` should be used as a label to exclude it from training
- `viewers_mean` should be filled with the median value

In [ ]:
# Drop the null values in viewers_mean and rank
summary_df = summary_df.dropna(subset=['viewers_mean', 'rank'])

summary_df.info()

### Results

**Summary:** Summary Data Frame
The current `summary_df` now has reasonable column names and column types, and no null values in the retained rows after removing rows with missing `viewers_mean` and `rank`. 


## Table: Challenges


### Initial CSV exploration

#### Relevance:
- Examines outcomes when contestants and tribes compete; these outcomes influence decisions when tribes vote contestants out.

#### Size:
- 5023 rows
- 8 columns
- 314.1 KB

#### Column descriptions (English):
- **season_name**: The season's name
- **season**: The season number
- **episode**: The episode number within the season
- **title**: The title of the episode
- **day**: The running day count since the game began
- **challenge_type**: Reward (a prize) or immunity (protection from being voted out)
- **winners**: The name of the contestant who won the challenge
- **winning_tribe**: The name of the tribe that won the challenge

**Note**: Actual column names and data types are shown below, along with a view of the first five rows.


#### Raw inspection


In [ ]:
print(challenges_df.info())
print(challenges_df.head())

# Challenge nulls before cleaning
challenge_nulls_before = challenges_df[["winners", "winning_tribe"]].isna().sum()
print("Nulls before cleaning:")
print(challenge_nulls_before)

### Cleaning Needs / Methods

The following code cell houses all the challenge-row cleaning rules for `challenges_df`.


In [ ]:
# Clean Version
challenges_cleaned_df = challenges_df.copy()

# Dropping rows that are placeholder / non-challenge rows
drop_mask = (
    # Survivor: Island of the Idols, episode 12
    ((challenges_cleaned_df["season"] == 39) &
     (challenges_cleaned_df["episode"] == 12) &
     (challenges_cleaned_df["day"] == 36) &
     (challenges_cleaned_df["challenge_type"] == "immunity") &
     (challenges_cleaned_df["winners"].isna())) |

    # Survivor: David vs. Goliath, episode 4
    ((challenges_cleaned_df["season"] == 37) &
     (challenges_cleaned_df["episode"] == 4) &
     (challenges_cleaned_df["day"] == 10) &
     (challenges_cleaned_df["challenge_type"] == "immunity") &
     (challenges_cleaned_df["winners"].isna()))
)

challenges_cleaned_df = challenges_cleaned_df.loc[~drop_mask].copy()

# Filling in rows where there truly was no immunity winner
no_winner_cases = [
    (32, 13),  # Kaoh Rong (Joe evacuated)
    (24, 6),   # One World (Colton evacuated)
    (21, 12),  # Nicaragua (NaOnka and Kelly quit)
    (19, 6),   # Samoa (Russell Swan evacuated)
    (12, 11),  # Panama (Bruce evacuated)
    (8, 3),    # All-Stars (Jenna quit)
    (8, 6),    # All-Stars (Sue quit)
    (2, 6)     # Australian Outback (Michael evacuated)
]

for season_num, episode_num in no_winner_cases:
    mask = (
        (challenges_cleaned_df["season"] == season_num) &
        (challenges_cleaned_df["episode"] == episode_num) &
        (challenges_cleaned_df["challenge_type"] == "immunity") &
        (challenges_cleaned_df["winners"].isna())
    )

    challenges_cleaned_df.loc[mask, "winners"] = "No challenge winner"
    challenges_cleaned_df.loc[mask, "winning_tribe"] = "Not applicable"

# If a winner exists but winning_tribe is missing, then tribe winner does not apply
tribe_not_applicable_mask = (
    challenges_cleaned_df["winners"].notna() &
    challenges_cleaned_df["winning_tribe"].isna()
)

challenges_cleaned_df.loc[tribe_not_applicable_mask, "winning_tribe"] = "Not applicable"

# Fixing Survivor: Blood vs. Water, episode 1
# Galang won the combined immunity/reward challenge, so restore the missing immunity winners
bvw_bad_immunity_rows = (
    (challenges_cleaned_df["season"] == 27) &
    (challenges_cleaned_df["episode"] == 1) &
    (challenges_cleaned_df["day"] == 1) &
    (challenges_cleaned_df["challenge_type"] == "immunity") &
    (challenges_cleaned_df["winners"].isna())
)

challenges_cleaned_df = challenges_cleaned_df.loc[~bvw_bad_immunity_rows].copy()

galang_members = [
    "Aras",
    "Colton",
    "Gervase",
    "Kat",
    "Laura B.",
    "Laura M.",
    "Monica",
    "Tina",
    "Tyson"
]

bvw_immunity_rows = pd.DataFrame(
    [
        {
            "season_name": "Survivor: Blood vs. Water",
            "season": 27,
            "episode": 1,
            "title": "Blood Is Thicker Than Anything",
            "day": 1,
            "challenge_type": "immunity",
            "winners": member,
            "winning_tribe": "Galang"
        }
        for member in galang_members
    ]
)

challenges_cleaned_df = pd.concat(
    [challenges_cleaned_df, bvw_immunity_rows],
    ignore_index=True
)

# Drop the last 3 blank reward placeholder rows
final_placeholder_rows = (
    ((challenges_cleaned_df["season"] == 22) &
     (challenges_cleaned_df["episode"] == 14) &
     (challenges_cleaned_df["day"] == 38) &
     (challenges_cleaned_df["challenge_type"] == "reward") &
     (challenges_cleaned_df["winners"].isna())) |

    ((challenges_cleaned_df["season"] == 16) &
     (challenges_cleaned_df["episode"] == 1) &
     (challenges_cleaned_df["day"] == 3) &
     (challenges_cleaned_df["challenge_type"] == "reward") &
     (challenges_cleaned_df["winners"].isna())) |

    ((challenges_cleaned_df["season"] == 13) &
     (challenges_cleaned_df["episode"] == 6) &
     (challenges_cleaned_df["day"] == 15) &
     (challenges_cleaned_df["challenge_type"] == "reward") &
     (challenges_cleaned_df["winners"].isna()))
)

challenges_cleaned_df = challenges_cleaned_df.loc[~final_placeholder_rows].copy()

# Sort the cleaned df so the column names make sense
challenges_cleaned_df = challenges_cleaned_df.sort_values(
    by=["season", "episode", "day", "challenge_type", "winners"]
).reset_index(drop=True)

# Challenge nulls after cleaning
challenge_nulls_after = challenges_cleaned_df[["winners", "winning_tribe"]].isna().sum()
print("\nChallenge nulls after cleaning:")
print(challenge_nulls_after)

print(challenges_cleaned_df.info())
print(challenges_cleaned_df.head())

### Results

See the printed null counts and `challenges_cleaned_df.info()` / `head()` output above for this run.


## Table: Castaways


### Initial CSV exploration

#### Context

#### Relevance:
- Personal data on each contestant and their performance.
- Useful for analyzing how traits relate to outcomes.

#### Size:
- 744 rows
- 18 columns
- 104.8 KB

#### Column descriptions (English):
- **season_name**: The name of the season
- **season**: The season number
- **full_name**: The contestant's full name
- **castaway**: The castaway's (contestant's) first name
- **age**: The contestant's age
- **city**: The city the contestant is from
- **state**: The state the contestant is from
- **personality_type**: Their personality description
- **day**: The day of the season (running count)
- **order**: Finish order for the season (larger values mean the contestant lasted longer)
- **result**: When they were voted out (string description of the order column)
- **jury_status**: If and when the player made the jury
- **original_tribe**: The tribe they started on
- **swapped_tribe**: The tribe they swapped to
- **swapped_tribe2**: The tribe they swapped to a second time
- **merged_tribe**: The merged tribe name
- **total_votes_received**: Number of votes cast against the contestant
- **immunity_idols_won**: Number of immunity idols won by the contestant

**Note**: Actual column names and data types are shown below, along with a view of the first five rows.


#### Raw inspection


In [ ]:
castaways_df.info()
castaways_df.head()

#### Raw inspection (continued)

The following information is used to determine the basic information about the dataset that is needed, and explained at the top of this section.


In [ ]:
# Shape and column names
print("SHAPE")
print(castaways_df.shape)

print("\nCOLUMNS")
print(castaways_df.columns.tolist())

# Data types
print("\nDATA TYPES")
print(castaways_df.dtypes)

# Missing values
print("\nMISSING VALUES")
print(castaways_df.isnull().sum())


### Cleaning needs

#### Data cleaning: Fixing null values

Based on the above cell we find the following columns have null values:
| Column Name | Null Count |
|--------|-----------|
| personality_type | 3 |
| jury_status | 405 |
| original_tribe | 2 |
| swapped_tribe | 284 |
| swapped_tribe2 | 683 |
| merged_tribe | 300 |


### Methods

The following code cells implement the castaways cleaning steps on `clean_castaways_df`.


In [ ]:
#first we create a copy of the dataframe for editing
clean_castaways_df = castaways_df.copy()

#### personality_type
First we will fix the missing personality types. Since we are only missing three, we could just drop these rows, but we may want other information on these players, so we will fill them with 'UNKNOWN' instead.

In [ ]:
print(f"Before: \n")
print(clean_castaways_df['personality_type'].info())

In [ ]:
print(clean_castaways_df['personality_type'].unique())
clean_castaways_df['personality_type'] = clean_castaways_df['personality_type'].fillna('UNKNOWN')

In [ ]:
print("After: \n")
print(clean_castaways_df['personality_type'].info())

#### jury_status

In [ ]:
# See all unique values in the column
print(f"Before: \n")
print(clean_castaways_df['jury_status'].info())

In [ ]:
print(castaways_df['jury_status'].unique())

After viewing this we can understand that this column tells us what member of the jury a castaway is. The players eliminated earlier in the season don't make it on the jury, which explains why there are so many null values. After some consideration we will replace all null values with "non jury member" to follow the naming convention. 

In [ ]:
clean_castaways_df['jury_status'] = clean_castaways_df['jury_status'].fillna('non jury member')

In [ ]:
print("After: \n")
print(clean_castaways_df['jury_status'].info())

#### original_tribe
Since there are only two null values in this column, we will investigate which two castaways have the null values to determine if we should drop the rows or replace the null with a value.

In [ ]:
print(f"Before: \n")
print(clean_castaways_df['original_tribe'].info())

In [ ]:
#output the two null rows
print(castaways_df[castaways_df['original_tribe'].isnull()])

After some googling I found this explanation from a Reddit post: "For those of you who don’t know, in the very first episode of Palau (Season 10), Jonathan Libby and Wanda Shirk were eliminated before tribes were even officially formed." Based on this we can conclude that we can drop these rows from the dataframe since they are inconsequential to understanding themes in the show. 

In [ ]:
clean_castaways_df = clean_castaways_df.dropna(subset=['original_tribe'])


In [ ]:
print("After: \n")
clean_castaways_df['original_tribe'].info()

#### swapped_tribe, swapped_tribe2
We will handle swapped_tribe and swapped_tribe2 together since they contain the same type of information, but some castaways swap tribes once, twice, or never.

In [ ]:
print(f"Before: \n")
print(clean_castaways_df['swapped_tribe'].info())
print(clean_castaways_df['swapped_tribe2'].info())

In [ ]:
print(clean_castaways_df['swapped_tribe'].unique())
print(clean_castaways_df['swapped_tribe2'].unique())

From this output and some general knowledge, we can understand that these columns tell us what tribe a castaway switched to. However, not all players switch tribes, and especially most players don't switch tribes twice, but they are still important to the story the data is telling us. We will fill these with "Not Applicable".

In [ ]:
swap_columns = ['swapped_tribe', 'swapped_tribe2']
clean_castaways_df[swap_columns] = clean_castaways_df[swap_columns].fillna('Not Applicable')

In [ ]:
print("After: \n")
clean_castaways_df[swap_columns].info()

#### merged_tribe

In [ ]:
print(f"Before: \n")
print(clean_castaways_df['merged_tribe'].info())

In [ ]:
print(clean_castaways_df['merged_tribe'].unique())
print(f"Number of merged tribe names: {clean_castaways_df['merged_tribe'].nunique()}")

After viewing this list of the merged tribe names we now understand that this column represents the name of the merged tribe a castaway was in. Part way through the season all of the tribes get merged into one tribe, and they create a new name for the tribe. Since around half of the players each season get eliminated before the merge, said players have null values. We will fill this with 'Not Applicable' since we still want the data on the players who don't make it to the merge.

In [ ]:
clean_castaways_df['merged_tribe'] = clean_castaways_df['merged_tribe'].fillna('Not Applicable')

In [ ]:
print("After: \n")
clean_castaways_df['merged_tribe'].info()

### Results


#### The data for the Castaways dataframe is now cleaned.

In [ ]:
clean_castaways_df.info()

## Table: Viewers


### Initial CSV exploration

#### Context

#### Relevance:
- Useful for gauging interest in Survivor.
- May reveal relationships between public interest and contestant performance.

#### Size:
- 596 rows
- 9 columns
- 42 KB

#### Column descriptions (English):
- **season_name**: The name of the season
- **season**: The season number
- **episode_number_overall**: The episode number across all seasons
- **episode**: The episode number within this season
- **title**: The title of the episode
- **episode_date**: The date the episode aired
- **viewers**: The number of viewers, in millions
- **rating_18_49**: Percentage of TV households in the 18-49 demographic that watched Survivor
- **share_18_49**: Among 18-49 viewers watching TV during the time slot, the percentage who watched Survivor

**Note**: Actual column names and data types are shown below, along with a view of the first five rows.


#### Raw inspection


In [ ]:
viewers_df.info()
viewers_df.head()

### Cleaning needs

_No additional cleaning steps in this notebook for `viewers_df` yet._ Fill this in when you add wrangling for this table.


### Methods

None yet for this table in this notebook.


### Results

Re-run `viewers_df.info()` after future cleaning; for now the raw inspection above is the check.


## Table: Jury votes


### Initial CSV exploration

#### Context

#### Relevance:
- Captures how jury members voted at Final Tribal Council.
- May reveal how personalities affect outcomes in the end.

#### Size:
- 909 rows
- 5 columns
- 35.6 KB

#### Column descriptions (English):
- **season_name**: The name of the season
- **season**: The season number
- **castaway**: The juror casting votes
- **finalist**: A finalist for the season who can receive jury votes
- **vote**: Whether the juror voted for this finalist (1 = yes, 0 = no)

**Note**: Actual column names and data types are shown below, along with a view of the first five rows.


#### Raw inspection


In [ ]:
jury_votes_df.info()
jury_votes_df.head()

### Cleaning needs

No cleaning steps for this dataframe yet. There are no nulls, appropriate column names, and column types.


## Sources Used

- **Tool:** Cursor (agent)
- **How used:** Notebook structure only—reordered cells (Summary cleaning moved next to Summary inspection), added section headings (B-style spine and C-style subsections), and stub Cleaning sections for viewers/jury; no changes to data-cleaning logic or substantive analysis text.
- **Scope:** Refinement / presentation only—not full draft generation of graded analysis.

- **Second pass (same tool):** Added the **Initial CSV exploration** / **Cleaning needs** / **Methods** / **Results** heading scheme under each `## Table:` section; heading and label edits only—no Python logic changes.
